# 🧱 강화학습 완전 기초 — DQN 으로 벽돌깨기

**AI CITY BUILDERS** · 다음 세대의 AI 자동화를 이끌 강화학습, 오늘은 그 첫걸음입니다.

오늘 할 일은 하나입니다. **아무것도 모르는 AI 가, 해 보고 · 점수를 받고 · 스스로 배워서 벽돌을 깨게 만든다.**

| 강화학습 단어 | 벽돌깨기에서는 |
|---|---|
| **상태** (state) | 지금 게임 화면 |
| **행동** (action) | 가만히 · 발사 · 오른쪽 · 왼쪽 |
| **보상** (reward) | 벽돌을 깨면 받는 점수 |
| **다음 상태** (next state) | 행동한 뒤 바뀐 화면 |
| **정책** (policy) | 화면을 보고 행동을 고르는 두뇌 |

👉 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 켜 주세요. (없어도 돌지만 느립니다)
👉 칸을 위에서부터 하나씩 ▶ 누르면 됩니다.

## 0. 준비 — 게임과 도구 설치 (1분)

In [ ]:
!pip install -q "gymnasium[atari]>=1.0" "ale-py>=0.9" imageio-ffmpeg
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import random, time, collections, base64, os, glob
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym, ale_py, imageio
import matplotlib, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display
gym.register_envs(ale_py)

# 한글 글꼴 (그래프·영상 글씨)
글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    matplotlib.font_manager.fontManager.addfont(글꼴경로)
    plt.rcParams['font.family'] = matplotlib.font_manager.FontProperties(fname=글꼴경로).get_name()
plt.rcParams['axes.unicode_minus'] = False

장치 = 'cuda' if torch.cuda.is_available() else 'cpu'
print('✅ 준비 끝 · 계산 장치:', 장치, '(cuda = GPU)')

In [ ]:
# 영상 보기 도우미: 화면 여러 장 → mp4 → 코랩 안에서 재생
def 영상보기(화면들, fps=30, 너비=480):
    이름 = f'video_{int(time.time()*1000)}.mp4'
    imageio.mimsave(이름, 화면들, fps=fps, macro_block_size=1)
    데이터 = base64.b64encode(open(이름, 'rb').read()).decode()
    display(HTML(f'<video width="{너비}" controls autoplay loop muted><source src="data:video/mp4;base64,{데이터}" type="video/mp4"></video>'))
    return 이름

## 1. 게임 열어 보기 — 상태와 행동

게임 화면이 곧 **상태**입니다. AI 는 이 숫자 덩어리(210 × 160 × 3 = 100,800개)를 보고 행동을 골라야 합니다.

In [ ]:
env = gym.make('ALE/Breakout-v5', render_mode='rgb_array')
상태, 정보 = env.reset(seed=0)
행동이름 = ['가만히', '발사', '오른쪽', '왼쪽']

print('상태(화면) 모양 :', 상태.shape, '→ 세로 210 · 가로 160 · 색 3')
print('상태 안의 숫자  :', 상태.size, '개 (0~255)')
print('할 수 있는 행동 :', env.action_space.n, '개 →', env.unwrapped.get_action_meanings(), '=', 행동이름)
print('남은 목숨       :', 정보['lives'])

plt.figure(figsize=(3, 4)); plt.imshow(상태); plt.title('AI 가 보는 상태 = 화면'); plt.axis('off'); plt.show()

## 2. 아무것도 모르는 AI — 무작위로 해 보기

아직 배우지 않은 AI 는 행동을 **아무거나** 고릅니다. 한 걸음마다 **상태 → 행동 → 보상 → 다음 상태** 가 어떻게 흘러가는지 print 로 봅시다.

In [ ]:
상태, 정보 = env.reset(seed=1)
화면들 = [env.render()]
총점, 걸음 = 0, 0
print(f"{'걸음':>4} | {'행동':<4} | {'보상':>4} | {'총점':>4} | 목숨 | 상태 요약 (화면 밝기 평균)")
print('-' * 62)
while True:
    행동 = env.action_space.sample()                     # ← 아무거나!
    다음상태, 보상, 끝남, 잘림, 정보 = env.step(행동)
    총점 += 보상; 걸음 += 1
    if 걸음 <= 15 or 보상 > 0:                             # 처음 15걸음 + 점수 난 순간만 찍는다
        print(f"{걸음:>4} | {행동이름[행동]:<4} | {보상:>+4.0f} | {총점:>4.0f} | {정보['lives']:>4} | {상태.mean():.2f} → {다음상태.mean():.2f}"
              + ('   ⭐ 벽돌 깼다!' if 보상 > 0 else ''))
    화면들.append(env.render())
    상태 = 다음상태
    if 끝남 or 잘림: break
print('-' * 62)
print(f'🎲 무작위 AI: {걸음}걸음 만에 게임 끝 · 점수 {총점:.0f}')
영상보기(화면들[:1500], fps=60)

대부분 공을 놓치고, 가끔 **운으로** 벽돌을 깹니다. 이제 이 AI 가 **보상을 보고 스스로 배우게** 만들겠습니다.

---
## 3. AI 의 눈 — 상태를 보기 좋게 다듬기

DQN 논문(딥마인드, 2015) 과 똑같이 다듬습니다.
- **흑백 84×84** 로 줄이기 (색은 필요 없다)
- **4장씩 겹쳐 보기** — 한 장만 보면 공이 어느 쪽으로 가는지 모른다
- **4프레임마다 한 번 결정** (사람도 매 순간 버튼을 바꾸지 않는다)
- 공이 없으면 자동으로 **발사**

In [ ]:
class 발사도우미(gym.Wrapper):
    """공이 없으면 게임이 멈춘다. 시작할 때와 목숨을 잃었을 때 자동으로 「발사」를 눌러 준다."""
    def reset(self, **kw):
        o, i = self.env.reset(**kw); o, _, _, _, i = self.env.step(1); self.목숨 = i['lives']; return o, i
    def step(self, a):
        o, r, term, trunc, i = self.env.step(a)
        i['목숨잃음'] = i['lives'] < self.목숨
        if i['목숨잃음'] and not (term or trunc):
            self.목숨 = i['lives']; o, r2, term, trunc, i2 = self.env.step(1); r += r2; i2['목숨잃음'] = True; i = i2
        return o, r, term, trunc, i

def 환경만들기():
    e = gym.make('ALE/Breakout-v5', frameskip=1, repeat_action_probability=0.0, render_mode='rgb_array')
    e = gym.wrappers.AtariPreprocessing(e, frame_skip=4, screen_size=84, grayscale_obs=True, noop_max=30)
    e = 발사도우미(e)
    return gym.wrappers.FrameStackObservation(e, 4)

env = 환경만들기()
상태, _ = env.reset(seed=0)
for _ in range(12): 상태, *_ = env.step(2)
상태 = np.array(상태)
print('다듬은 상태 모양:', 상태.shape, '→ 화면 4장 × 84 × 84 =', 상태.size, '개 숫자')
fig, ax = plt.subplots(1, 4, figsize=(10, 3))
for k in range(4):
    ax[k].imshow(상태[k], cmap='gray'); ax[k].set_title(f'{k+1}번째 (과거→현재)'); ax[k].axis('off')
plt.suptitle('AI 가 한 번에 보는 상태 = 화면 4장'); plt.show()

## 4. AI 의 두뇌 — Q 네트워크

**Q값** = 「지금 이 상태에서 이 행동을 하면, 앞으로 점수를 얼마나 받을까?」 의 예상치.

두뇌는 화면 4장을 받아 **행동 4개의 Q값** 을 내놓습니다. AI 는 **Q값이 가장 큰 행동** 을 고릅니다.

```
화면 4장 ──▶ [눈: 합성곱 3층] ──▶ [판단: 512] ──▶ Q(가만히) Q(발사) Q(오른쪽) Q(왼쪽)
```

In [ ]:
class Q두뇌(nn.Module):
    """화면 4장 → 행동 4개의 Q값 (DQN 논문 구조)"""
    def __init__(self, n=4):
        super().__init__()
        self.눈 = nn.Sequential(nn.Conv2d(4, 32, 8, 4), nn.ReLU(), nn.Conv2d(32, 64, 4, 2), nn.ReLU(),
                                nn.Conv2d(64, 64, 3, 1), nn.ReLU(), nn.Flatten())
        self.판단 = nn.Sequential(nn.Linear(3136, 512), nn.ReLU(), nn.Linear(512, n))
    def forward(self, x): return self.판단(self.눈(x.float() / 255.0))

q = Q두뇌().to(장치)
print('두뇌 크기:', sum(p.numel() for p in q.parameters()), '개 숫자(가중치)')
with torch.no_grad():
    Q값 = q(torch.as_tensor(상태, device=장치).unsqueeze(0))[0].cpu().numpy()
for n, v in zip(행동이름, Q값): print(f'  Q({n}) = {v:+.4f}')
print('→ 고를 행동:', 행동이름[int(Q값.argmax())], '  (아직 아무것도 안 배워서 Q값이 전부 0 근처 · 의미 없음)')

## 5. DQN 의 세 가지 비밀

| 비밀 | 한 줄 설명 | 코드 |
|---|---|---|
| ① **탐험 ε** | 처음엔 거의 아무거나(ε=1), 점점 두뇌를 믿는다(ε=0.05) | `엡실론()` |
| ② **기억 (리플레이 버퍼)** | 겪은 일 (상태·행동·보상·다음 상태) 을 쌓아 두고 **무작위로 꺼내 복습** | `기억` |
| ③ **목표 두뇌** | 정답을 만드는 두뇌는 **1만 걸음마다 한 번만** 복사 → 흔들리지 않게 | `목표` |

**배우는 공식 (벨만 방정식)**

> 목표값 = **보상** + 0.99 × (목표 두뇌가 본 **다음 상태의 가장 큰 Q**)
>
> 오차 = 목표값 − 지금 두뇌의 Q(상태, 행동) → 이 오차를 줄이는 쪽으로 두뇌를 고친다

In [ ]:
def 엡실론(걸음, 시작=1.0, 끝=0.05, 기간=100_000):
    """① 탐험: 걸음이 늘수록 ε 이 줄어든다"""
    return max(끝, 시작 - (시작 - 끝) * 걸음 / 기간)

class 기억:
    """② 리플레이 버퍼. 화면 묶음만 저장하고, 다음 화면은 바로 다음 칸에서 꺼낸다 (메모리 절약)."""
    def __init__(self, 크기):
        self.크기 = 크기; self.화면 = np.zeros((크기, 4, 84, 84), np.uint8); self.행동 = np.zeros(크기, np.int64)
        self.보상 = np.zeros(크기, np.float32); self.끝 = np.zeros(크기, np.float32); self.칸 = 0; self.찬 = 0
    def 넣기(self, s, a, r, d):
        self.화면[self.칸] = s; self.행동[self.칸] = a; self.보상[self.칸] = r; self.끝[self.칸] = d
        self.칸 = (self.칸 + 1) % self.크기; self.찬 = min(self.찬 + 1, self.크기)
    def 꺼내기(self, n):
        idx = np.random.randint(0, self.찬 - 1, size=n); idx = idx[(idx + 1) % self.크기 != self.칸]
        다음 = (idx + 1) % self.크기
        return self.화면[idx], self.행동[idx], self.보상[idx], self.화면[다음], self.끝[idx]

for 걸음 in [0, 25_000, 50_000, 100_000, 300_000]:
    print(f'{걸음:>7}걸음째 ε = {엡실론(걸음):.2f} → 아무거나 할 확률 {엡실론(걸음)*100:.0f}%')

## 6. 학습 시작 🚀

- `총걸음` 기본 **30만 걸음** (T4 GPU 로 약 10~15분). 시간이 있으면 100만 걸음까지 늘려 보세요.
- 1만 걸음마다 **현재 상태·고른 행동·Q값·보상** 을 print 로 보여 줍니다.
- 벽돌깨기는 오래 걸리는 게임입니다. 30만 걸음이면 「공을 쫓아가기 시작」 하는 정도이고, 잘하려면 수백만 걸음이 필요합니다. → **8번 칸에서 미리 오래 학습시킨 두뇌** 를 불러와 비교합니다.

In [ ]:
총걸음 = 300_000          # ← 시간이 있으면 1_000_000 으로
시작학습 = 20_000          # 이만큼은 기억만 쌓고 배우지 않는다
random.seed(0); np.random.seed(0); torch.manual_seed(0)

env = 환경만들기(); q = Q두뇌().to(장치); 목표 = Q두뇌().to(장치); 목표.load_state_dict(q.state_dict())
opt = torch.optim.Adam(q.parameters(), lr=1e-4); 메모리 = 기억(100_000)
s, _ = env.reset(seed=0); s = np.array(s); 점수 = 0; 판점수 = []; 판 = 0; 기록 = []; t0 = time.time()

for 걸음 in range(1, 총걸음 + 1):
    eps = 엡실론(걸음)
    with torch.no_grad(): Q값 = q(torch.as_tensor(s, device=장치).unsqueeze(0))[0].cpu().numpy()
    탐험 = random.random() < eps
    a = random.randrange(4) if 탐험 else int(Q값.argmax())                   # ① 탐험 or 두뇌

    s2, r, term, trunc, info = env.step(a); s2 = np.array(s2); 점수 += r
    끝 = float(term or info.get('목숨잃음', False))
    메모리.넣기(s, a, float(np.sign(r)), 끝)                                  # ② 기억에 쌓기

    if 걸음 % 10_000 == 0:                                                   # 👀 지금 AI 머릿속 들여다보기
        평균 = np.mean(판점수[-20:]) if 판점수 else 0
        Qtxt = ' '.join(f'{n}{v:+.2f}' for n, v in zip(행동이름, Q값))
        print(f'[{걸음:>7}걸음 | {(time.time()-t0)/60:4.1f}분 | ε {eps:.2f}] 최근 20판 평균 {평균:4.1f}점 | 판 {판}')
        print(f'    상태: 화면 4장(공·판 위치) → Q값 [{Qtxt}] → 행동 「{행동이름[a]}」{"(탐험!)" if 탐험 else "(두뇌 선택)"} → 보상 {r:+.0f}')
        기록.append((걸음, 평균))

    s = s2
    if term or trunc:
        판점수.append(점수); 판 += 1; 점수 = 0; s, _ = env.reset(); s = np.array(s)

    if 걸음 > 시작학습 and 걸음 % 4 == 0:                                   # 복습: 기억에서 32개 꺼내 배우기
        bs, ba, br, bs2, bd = 메모리.꺼내기(32)
        bs = torch.as_tensor(bs, device=장치); bs2 = torch.as_tensor(bs2, device=장치)
        ba = torch.as_tensor(ba, device=장치); br = torch.as_tensor(br, device=장치); bd = torch.as_tensor(bd, device=장치)
        with torch.no_grad(): 목표값 = br + 0.99 * 목표(bs2).max(1).values * (1 - bd)   # ③ 목표 두뇌가 정답을 만든다
        예측 = q(bs).gather(1, ba.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(예측, 목표값)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q.parameters(), 10); opt.step()
    if 걸음 % 10_000 == 0: 목표.load_state_dict(q.state_dict())             # ③ 가끔만 복사

torch.save(q.state_dict(), '내_두뇌.pt')
print(f'\n✅ 학습 끝! {(time.time()-t0)/60:.1f}분 · {판}판 · 두뇌를 내_두뇌.pt 로 저장')

## 7. 얼마나 배웠나 — 점수 곡선

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(판점수, alpha=.25, label='한 판 점수')
if len(판점수) >= 20:
    plt.plot(range(19, len(판점수)), np.convolve(판점수, np.ones(20) / 20, 'valid'), lw=2.5, label='최근 20판 평균')
plt.xlabel('판'); plt.ylabel('점수'); plt.title('AI 가 스스로 배운 기록'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. AI 가 하는 걸 보자 — 영상 + 머릿속 (Q값 막대)

영상 오른쪽에 **매 순간 AI 가 계산한 Q값** 과 **고른 행동** 이 나옵니다. 파란 막대 = 고른 행동.

In [ ]:
def 글꼴(크기):
    try: return ImageFont.truetype(글꼴경로, 크기)
    except Exception: return ImageFont.load_default()

def 머릿속그리기(화면, Q값, 행동, 걸음, 총점, 보상):
    """게임 화면 옆에 Q값 막대를 붙인 그림 한 장"""
    게임 = Image.fromarray(화면).resize((320, 420), Image.NEAREST)
    판 = Image.new('RGB', (560, 420), (14, 18, 28)); 판.paste(게임, (0, 0)); d = ImageDraw.Draw(판)
    d.text((340, 18), f'걸음 {걸음}', fill=(200, 210, 225), font=글꼴(18))
    d.text((340, 44), f'점수 {총점:.0f}', fill=(255, 214, 102), font=글꼴(26))
    d.text((340, 86), 'AI 머릿속 (Q값)', fill=(150, 160, 180), font=글꼴(15))
    lo, hi = min(Q값.min(), 0), max(Q값.max(), 0.01)
    for k, (n, v) in enumerate(zip(행동이름, Q값)):
        y = 116 + k * 56; 길이 = int(170 * (v - lo) / (hi - lo + 1e-9))
        색 = (56, 189, 248) if k == 행동 else (71, 85, 105)
        d.text((340, y), n, fill=(230, 235, 245) if k == 행동 else (150, 160, 180), font=글꼴(16))
        d.rectangle([340, y + 24, 340 + max(길이, 2), y + 42], fill=색)
        d.text((520 - 4, y + 24), f'{v:.2f}', fill=(200, 210, 225), font=글꼴(13), anchor='ra')
    d.text((340, 352), f'▶ 행동: {행동이름[행동]}', fill=(56, 189, 248), font=글꼴(20))
    if 보상 > 0: d.text((340, 384), f'⭐ 보상 +{보상:.0f}', fill=(255, 214, 102), font=글꼴(18))
    return np.array(판)

def 두뇌로플레이(두뇌, 씨앗=0, 최대걸음=3000, 찍기=12):
    e = 환경만들기(); s, _ = e.reset(seed=씨앗); s = np.array(s); 총점 = 0; 그림들 = []
    print(f"{'걸음':>4} | Q(가만히) Q(발사) Q(오른쪽) Q(왼쪽) | 행동   | 보상 | 총점")
    for 걸음 in range(1, 최대걸음 + 1):
        with torch.no_grad(): Q값 = 두뇌(torch.as_tensor(s, device=장치).unsqueeze(0))[0].cpu().numpy()
        a = int(Q값.argmax()) if random.random() > 0.01 else random.randrange(4)
        s, r, term, trunc, _ = e.step(a); s = np.array(s); 총점 += r
        if 걸음 <= 찍기 or r > 0 and 걸음 < 400:
            print(f"{걸음:>4} | " + '  '.join(f'{v:+7.3f}' for v in Q값) + f" | {행동이름[a]:<5}| {r:>+4.0f} | {총점:>4.0f}")
        그림들.append(머릿속그리기(e.render(), Q값, a, 걸음, 총점, r))
        if term or trunc: break
    print(f'🎮 게임 끝: {걸음}걸음 · 점수 {총점:.0f}')
    return 그림들, 총점

그림들, _ = 두뇌로플레이(q)
영상보기(그림들, fps=20, 너비=560)

## 9. 미리 오래 학습시킨 두뇌와 비교

AI CITY BUILDERS 가 **수백만 걸음** 학습시켜 둔 두뇌를 불러옵니다. 같은 코드, 같은 두뇌 구조 — **다른 건 학습한 시간뿐** 입니다.

In [ ]:
주소 = 'https://github.com/wonseokjung/aicb-colab/raw/main/models/breakout_dqn.pt'
!wget -q -O 미리학습_두뇌.pt "{주소}"
고수 = Q두뇌().to(장치)
try:
    고수.load_state_dict(torch.load('미리학습_두뇌.pt', map_location=장치)); 고수.eval()
    그림들, 점수 = 두뇌로플레이(고수, 씨앗=3)
    영상보기(그림들, fps=20, 너비=560)
except Exception as e:
    print('미리 학습한 두뇌를 아직 받지 못했습니다:', e)

## 10. 직접 바꿔 보기 🧪

6번 칸의 숫자를 바꾸고 다시 돌려 보세요. 무엇이 달라지나요?

| 바꿀 것 | 지금 | 해 볼 것 | 생각해 볼 질문 |
|---|---|---|---|
| `엡실론(... 기간=)` | 100,000 | 10,000 | 탐험을 너무 빨리 멈추면? |
| `0.99` (감마) | 0.99 | 0.5 | 먼 미래의 보상을 무시하면? |
| `목표.load_state_dict` 주기 | 10,000 | 1 (매번) | 목표 두뇌가 없으면? |
| `기억(100_000)` | 100,000 | 1,000 | 기억이 짧으면? |

---
## 🏆 다음: 벽돌깨기 두뇌 대회

아타리 벽돌깨기는 제대로 잘하려면 몇 시간이 걸립니다. 그래서 대회는 **규칙이 같은 미니 벽돌깨기** 로 합니다 (1분 학습).

👉 대회 코랩: **강화학습_벽돌깨기_대회.ipynb** (같은 저장소)
👉 AI 가 배우는 모습 보기: **aicitybuilders.com/dqn**
👉 순위표: **aicitybuilders.com/contest?c=brick**